# 02 · ephys anomaly-fix: resolve problems, then organise the files

Input: `01_ephys_meta-org/output/recordings.csv` and `flags.csv` (run notebook 01 first).

1. **Automatic resolution** of the two known acquisition problems, each logged:
   * *voltage recorded twice, no current* → baseline protocols (0 pA): **voltage only**; step / ramp protocols:
     **command waveform used as the current** (only when the telegraphed sensitivity is the expected 400 pA/V);
   * *gain error* → **rescale** by the standard amplifier gain that clearly brings the voltage to a normal RMP.
2. **Decision sheet** (`decisions.xlsx`, in this folder) for everything else, e.g. a protocol mismatch or a
   bias current noted in the recording book. You fill in the `decision` column; your decisions are never overwritten.
3. **Apply** decisions: each recording becomes *ready*, *pending* or *excluded*.
4. **Organise:** only *ready* recordings are copied into `output/protocol_sorted/<protocol>/`.
5. **Report:** `output/issues_report.md` / `.csv` lists every issue and how it was resolved;
   `output/analysis_ready.csv` tells the protocol analyses what to apply (current source, scaling, bias).

The original ABFs are never changed: corrections are written down, not applied to the files.

## Setup
`FRESH = True` clears `output/` (the decision sheet is kept).

In [ ]:
FRESH = True

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
from ephys_analysis import anomaly_fix as af
from ephys_analysis.utils import RunLog, find_repo_root, repo_path

REPO = find_repo_root()
cfg, meta_cfg, out = af.setup(REPO, fresh=FRESH)
log = RunLog(cfg, out)
recordings, flags = af.load_inputs(cfg)
print(f"{len(recordings)} recordings, {len(flags)} flags from 01")
display(flags.groupby(["severity", "flag"]).size().rename("n").to_frame())

## Step 1 · Automatic resolution
Rules are in `config.yaml` → `auto_resolve`. Anything that does not meet a rule is left for a decision.

In [ ]:
auto = af.auto_resolve(recordings, flags, cfg)
display(auto.merge(recordings[["recording_id", "abf_filename", "abf_family"]], on="recording_id")
            [["abf_filename", "abf_family", "flag", "action", "value", "how"]])

## Step 2 · Decision sheet
New issues are added to `decisions.xlsx` with a **suggested action**. Rows you have already filled in are kept as they are.

In [ ]:
sheet, n_new = af.update_decision_sheet(af.pending_flags(flags, auto), recordings, cfg)
sheet = af.validate_decisions(sheet, cfg)
print(f"{n_new} new rows; sheet: {repo_path(cfg, cfg['decision_sheet'])}")
todo = sheet[sheet["decision_status"] != "ok"]
display(todo[["abf_filename", "flag", "detail", "suggested_action", "suggested_value", "allowed_actions", "decision_status"]])

### ✋ Checkpoint
If rows are listed above, open **`decisions.xlsx`** (in this folder) and for each row fill in:

| column | what to enter |
|---|---|
| `decision` | one of `allowed_actions` (dropdown): e.g. `bias_current`, `relabel`, `accept`, `exclude` |
| `value` | for `bias_current`: pA (e.g. `-10`); `relabel`: protocol family; `rescale`: gain factor |
| `note` | why (required for `exclude`) |
| `decided_by`, `decided_on` | your initials and the date |

Save and **close** Excel, then re-run this notebook from *Step 2*. You can also continue now: undecided
recordings simply stay *pending* and are not copied.

## Step 3 · Apply decisions

In [ ]:
status, issues = af.resolve(recordings, flags, auto, sheet, cfg.get("noise_corrections"))
display(status["state"].value_counts().to_frame())
display(status.merge(recordings[["recording_id", "abf_filename"]], on="recording_id")
              [["abf_filename", "state", "family", "voltage_source", "current_source", "voltage_scale",
                "bias_current_pA", "corrections", "open_issues"]])

## Step 4 · Organise
Ready recordings are copied (SHA-256 verified) into `output/protocol_sorted/<protocol folder>/<recording day>/`.
Copies of recordings that are no longer ready are removed.

In [ ]:
manifest = af.organise(recordings, status, meta_cfg, out, log)
display(manifest.groupby(["destination_folder", "copy_status"]).size().rename("n").to_frame())

## Step 5 · Report

In [ ]:
paths = af.write_outputs(recordings, status, issues, manifest, out, log.run_id)
log.finish(paths)
display(Markdown(paths[2].read_text(encoding="utf-8")))

**Next:** `03_ephys_membrane-test/membrane-test.ipynb` (uses `output/analysis_ready.csv`).